#### CELDA 1: CONTROL DE ENTORNO E IMPORTACIÓN

In [8]:
import os
import datetime
import numpy as np
import pandas as pd

# Desactivar advertencias de asignación encadenada para un código de producción limpio
pd.options.mode.chained_assignment = None

# Configuración rígida de rutas relativas desde la carpeta notebooks/
RUTA_RAW = "../data/raw"
RUTA_PROCESSED = "../data/processed"

print("========================================================================")
print("             PIPELINE DE ETL INICIALIZADO - FISIMart S.A.C.             ")
print("========================================================================")
print(f"● Origen de Datos (Raw)      : {RUTA_RAW}/")
print(f"● Destino del Datamart      : {RUTA_PROCESSED}/")
print("========================================================================")

             PIPELINE DE ETL INICIALIZADO - FISIMart S.A.C.             
● Origen de Datos (Raw)      : ../data/raw/
● Destino del Datamart      : ../data/processed/


#### CELDA 2: EXTRACCIÓN Y DIAGNÓSTICO DE CALIDAD INICIAL (REPORTE ANTES DEL ETL)

In [9]:
print("Cargando set de datos crudos desde el Data Lake (Raw)...")

# Extracción de fuentes transaccionales crudas
clientes_raw = pd.read_csv(f"{RUTA_RAW}/clientes_raw.csv", sep=";", encoding="utf-8")
productos_raw = pd.read_csv(f"{RUTA_RAW}/productos_raw.csv", sep=";", encoding="utf-8")
tiendas_raw = pd.read_csv(f"{RUTA_RAW}/tiendas_raw.csv", sep=";", encoding="utf-8")
tiempo_raw = pd.read_csv(f"{RUTA_RAW}/tiempo_raw.csv", sep=";", encoding="utf-8")
promociones_raw = pd.read_csv(f"{RUTA_RAW}/promociones_raw.csv", sep=";", encoding="utf-8")
ventas_raw = pd.read_csv(f"{RUTA_RAW}/ventas_raw.csv", sep=";", encoding="utf-8")

print("\n" + "="*85)
print("         REPORTE DE AUDITORÍA DE CALIDAD INICIAL (PRE-ETL) - FISIMart S.A.C.       ")
print("="*85)

# 1. Volumen total de filas
print("📊 VOLUMETRÍA INICIAL DE FILAS:")
print(f"  • Clientes Crudos      : {len(clientes_raw)} filas")
print(f"  • Productos Crudos     : {len(productos_raw)} filas")
print(f"  • Tiendas Crudas       : {len(tiendas_raw)} filas")
print(f"  • Tiempo Crudo         : {len(tiempo_raw)} filas")
print(f"  • Promociones Crudas   : {len(promociones_raw)} filas")
print(f"  • Ventas Crudas (Facts): {len(ventas_raw)} filas")

# 2. Conteo exacto de nulos
print("\n🔍 VALORES FALTANTES IDENTIFICADOS (NULOS):")
print(f"  • Clientes -> 'distrito' nulos: {clientes_raw['distrito'].isna().sum()}")
print(f"  • Productos -> 'categoria' nulos: {productos_raw['categoria'].isna().sum()}")
print(f"  • Ventas -> 'descuento' nulos: {ventas_raw['descuento'].isna().sum()}")

# 3. Cantidad de duplicados
print("\n🚨 REDUNDANCIA DE DATOS EN HECHOS:")
print(f"  • Registros duplicados exactos en Ventas: {ventas_raw.duplicated().sum()}")

# 4. Claves huérfanas
set_productos_maestros = set(productos_raw["id_producto"])
claves_huerfanas_ventas = ventas_raw[~ventas_raw["id_producto"].isin(set_productos_maestros)]
print("\n⚠️  EVALUACIÓN DE INTEGRIDAD REFERENCIAL:")
print(f"  • Lineas de venta con claves huerfanas (id_producto no existente): {len(claves_huerfanas_ventas)}")
print("="*85)

Cargando set de datos crudos desde el Data Lake (Raw)...

         REPORTE DE AUDITORÍA DE CALIDAD INICIAL (PRE-ETL) - FISIMart S.A.C.       
📊 VOLUMETRÍA INICIAL DE FILAS:
  • Clientes Crudos      : 5000 filas
  • Productos Crudos     : 600 filas
  • Tiendas Crudas       : 20 filas
  • Tiempo Crudo         : 730 filas
  • Promociones Crudas   : 41 filas
  • Ventas Crudas (Facts): 50750 filas

🔍 VALORES FALTANTES IDENTIFICADOS (NULOS):
  • Clientes -> 'distrito' nulos: 242
  • Productos -> 'categoria' nulos: 27
  • Ventas -> 'descuento' nulos: 2297

🚨 REDUNDANCIA DE DATOS EN HECHOS:
  • Registros duplicados exactos en Ventas: 738

⚠️  EVALUACIÓN DE INTEGRIDAD REFERENCIAL:
  • Lineas de venta con claves huerfanas (id_producto no existente): 152


#### CELDA 3: PIPELINE DE TRANSFORMACIÓN (ETL) - PARTE 1: SANEAMIENTO DE DIMENSIONES

In [10]:
print("Iniciando Fase de Purificación en Dimensiones Maestras...")

# ---- PIPELINE DIM_PRODUCTO ----
df_producto_clean = productos_raw.copy()

# Remover espacios en blanco extremos
df_producto_clean["categoria"] = df_producto_clean["categoria"].str.strip()

# Homologar y estandarizar variaciones caóticas de texto a categorías limpias corporativas
def homologar_categoria(val):
    if pd.isna(val):
        return "No Especificado"
    val_clean = val.strip().lower()
    if "bebi" in val_clean:
        return "Bebidas"
    if "abarro" in val_clean:
        return "Abarrotes"
    if "tecnolo" in val_clean:
        return "Tecnologia"
    if "hogar" in val_clean:
        return "Hogar"
    if "cuidado" in val_clean or "perso" in val_clean:
        return "Cuidado Personal"
    return "Otros"

df_producto_clean["categoria"] = df_producto_clean["categoria"].apply(homologar_categoria)
df_producto_clean.drop_duplicates(subset=["id_producto"], inplace=True)


# ---- PIPELINE DIM_CLIENTE ----
df_cliente_clean = clientes_raw.copy()

# Imputación segura de distritos nulos
df_cliente_clean["distrito"] = df_cliente_clean["distrito"].fillna("No Especificado")

# Función vectorizada flexible para parsear formatos mixtos de fechas latentes en el Raw
def limpiar_fechas_mixtas(columna):
    # Intentar parsear los formatos comunes; los que fallen quedarán como NaT temporalmente
    fechas_transformadas = pd.to_datetime(columna, format="%Y-%m-%d", errors="coerce")
    fechas_transformadas = fechas_transformadas.fillna(pd.to_datetime(columna, format="%d/%m/%Y", errors="coerce"))
    
    # Manejar el formato de texto suizo/personalizado '2024/Dic/15'
    mascara_texto = fechas_transformadas.isna() & columna.notna()
    if mascara_texto.any():
        meses_dic = {"ene": "01", "feb": "02", "mar": "03", "abr": "04", "may": "05", "jun": "06",
                     "jul": "07", "ago": "08", "sep": "09", "oct": "10", "nov": "11", "dic": "12"}
        
        def corregir_formato_texto(val):
            try:
                partes = str(val).split("/")
                if len(partes) == 3:
                    anio = partes[0]
                    mes_str = partes[1].lower()[:3]
                    mes = meses_dic.get(mes_str, "12")
                    dia = partes[2].zfill(2)
                    return pd.to_datetime(f"{anio}-{mes}-{dia}", format="%Y-%m-%d")
            except Exception:
                return pd.NaT
            return pd.NaT
            
        fechas_transformadas[mascara_texto] = columna[mascara_texto].apply(corregir_formato_texto)
    
    # Si alguna fecha crítica queda vacía por daño colateral, se imputa con la fecha base del proyecto
    return fechas_transformadas.fillna(pd.Timestamp("2024-01-01"))

df_cliente_clean["fecha_alta"] = limpiar_fechas_mixtas(df_cliente_clean["fecha_alta"])
df_cliente_clean["fecha_nacimiento"] = pd.to_datetime(df_cliente_clean["fecha_nacimiento"], errors="coerce").fillna(pd.Timestamp("1990-01-01"))

print("✓ Saneamiento y estandarización de catálogos maestros ejecutado correctamente.")

Iniciando Fase de Purificación en Dimensiones Maestras...
✓ Saneamiento y estandarización de catálogos maestros ejecutado correctamente.


CELDA 5: INGENIERÍA DE CARACTERÍSTICAS (MÉTRICAS DERIVADAS)

In [11]:
print("Ejecutando limpieza dura e integridad sobre la matriz de hechos Ventas...")

df_ventas_clean = ventas_raw.copy()

# 1. Estandarización homogénea de la columna temporal 'fecha'
df_ventas_clean["fecha"] = limpiar_fechas_mixtas(df_ventas_clean["fecha"])

# 2. Tratamiento correctivo de nulos en la columna de descuentos comerciales
df_ventas_clean["descuento"] = df_ventas_clean["descuento"].fillna(0.0)

# 3. Eliminación definitiva de redundancia transaccional (Duplicados exactos)
df_ventas_clean.drop_duplicates(inplace=True)

# 4. Filtrado y remoción de Outliers operativos de negocio (cantidades absurdas o precios rotos)
df_ventas_clean = df_ventas_clean[
    (df_ventas_clean["cantidad"] < 500) & 
    (df_ventas_clean["precio_unitario"] > 0.0)
]

# 5. Blindaje de Integridad Referencial: Eliminación estricta de registros de claves huérfanas
set_productos_limpios = set(df_producto_clean["id_producto"])
df_ventas_clean = df_ventas_clean[df_ventas_clean["id_producto"].isin(set_productos_limpios)]

print(f"✓ Tabla de Hechos purificada de forma rigurosa. Volumetría neta resultante: {len(df_ventas_clean)} filas.")

Ejecutando limpieza dura e integridad sobre la matriz de hechos Ventas...
✓ Tabla de Hechos purificada de forma rigurosa. Volumetría neta resultante: 49860 filas.


#### CELDA 5: INGENIERÍA DE CARACTERÍSTICAS (MÉTRICAS DERIVADAS)

In [12]:
print("Calculando métricas y variables financieras vectorizadas...")

# 1. Cálculo del Importe Neto aplicando el factor de descuento comercial
df_ventas_clean["importe"] = round(
    df_ventas_clean["cantidad"] * df_ventas_clean["precio_unitario"] * (1.0 - df_ventas_clean["descuento"]), 2
)

# 2. Enriquecimiento: Traer el costo unitario base desde la dimensión Producto limpia
df_ventas_clean = df_ventas_clean.merge(
    df_producto_clean[["id_producto", "costo"]], 
    on="id_producto", 
    how="left"
)

# 3. Cálculo del Costo Total de la Línea de Venta
df_ventas_clean["costo_total"] = round(df_ventas_clean["cantidad"] * df_ventas_clean["costo"], 2)

# 4. Cálculo del Margen de Ganancia Neto de la transacción
df_ventas_clean["margen"] = round(df_ventas_clean["importe"] - df_ventas_clean["costo_total"], 2)

print("✓ Ingeniería de características completada. Métricas 'importe', 'costo_total' y 'margen' inyectadas.")

Calculando métricas y variables financieras vectorizadas...
✓ Ingeniería de características completada. Métricas 'importe', 'costo_total' y 'margen' inyectadas.


#### CELDA 6: CARGA Y PERSISTENCIA EN DATAMART PROCESADO

In [13]:
print("Estructurando esquemas estrella definitivos y exportando a zona limpia...")

# Selección estricta de variables finales según requerimiento arquitectónico corporativo
columnas_clientes = ["id_cliente", "nombre", "sexo", "fecha_nacimiento", "edad", "distrito", "fecha_alta", "segmento_programa", "ingresos_mensuales_aprox"]
columnas_productos = ["id_producto", "nombre", "categoria", "subcategoria", "marca", "precio_lista", "costo"]
columnas_hechos_ventas = ["id_venta", "fecha", "id_cliente", "id_producto", "id_tienda", "id_promocion", "cantidad", "precio_unitario", "descuento", "importe", "costo_total", "margen"]

Dim_Cliente = df_cliente_clean[columnas_clientes]
Dim_Producto = df_producto_clean[columnas_productos]
Fact_Ventas = df_ventas_clean[columnas_hechos_ventas]

# Persistencia final en la carpeta limpia ../data/processed/ con codificación estándar
Dim_Cliente.to_csv(f"{RUTA_PROCESSED}/Dim_Cliente.csv", sep=";", index=False, encoding="utf-8")
Dim_Producto.to_csv(f"{RUTA_PROCESSED}/Dim_Producto.csv", sep=";", index=False, encoding="utf-8")
Fact_Ventas.to_csv(f"{RUTA_PROCESSED}/Fact_Ventas.csv", sep=";", index=False, encoding="utf-8")

# Nota: Tienda, Tiempo y Promoción ya se guardaron allí por su estado nativo limpio en el paso 00.
print("✓ El Datamart en Esquema Estrella ha sido cargado con éxito en la ruta processed/.")

Estructurando esquemas estrella definitivos y exportando a zona limpia...
✓ El Datamart en Esquema Estrella ha sido cargado con éxito en la ruta processed/.


#### CELDA 7: REPORTE DE CALIDAD FINAL DE AUDITORÍA (POST-ETL)

In [14]:
print("="*85)
print("             REPORTE DE CALIDAD FINAL CERTIFICADO (POST-ETL) - FISIMart S.A.C.          ")
print("="*85)

mapeo_procesado = {
    "Dim_Cliente (Limpia)": Dim_Cliente,
    "Dim_Producto (Limpia)": Dim_Producto,
    "Fact_Ventas (Limpia)": Fact_Ventas
}

for titulo, df_final in mapeo_procesado.items():
    print(f"\n✅ DATAMART ENTIDAD: {titulo}")
    print(f"  • Filas utiles resultantes : {len(df_final)}")
    print(f"  • Tasa de Valores Nulos    : {(df_final.isna().sum().sum() / df_final.size) * 100:.2f}%")
    print(f"  • Registros Duplicados     : {df_final.duplicated().sum()}")

# Confirmación de la erradicación total de claves huérfanas
set_prod_finales = set(Dim_Producto["id_producto"])
huerfanos_finales = Fact_Ventas[~Fact_Ventas["id_producto"].isin(set_prod_finales)]

print("\n" + "-"*85)
print("🛡️  CERTIFICACIÓN DE INTEGRIDAD REFERENCIAL:")
print(f"  • Claves huerfanas residuales en Fact_Ventas: {len(huerfanos_finales)}")
print(f"  • Estado del Datamart                       : COMPLETO Y LISTO PARA CONEXIÓN A POWER BI")
print("="*85)

             REPORTE DE CALIDAD FINAL CERTIFICADO (POST-ETL) - FISIMart S.A.C.          

✅ DATAMART ENTIDAD: Dim_Cliente (Limpia)
  • Filas utiles resultantes : 5000
  • Tasa de Valores Nulos    : 0.00%
  • Registros Duplicados     : 0

✅ DATAMART ENTIDAD: Dim_Producto (Limpia)
  • Filas utiles resultantes : 600
  • Tasa de Valores Nulos    : 0.00%
  • Registros Duplicados     : 0

✅ DATAMART ENTIDAD: Fact_Ventas (Limpia)
  • Filas utiles resultantes : 49860
  • Tasa de Valores Nulos    : 0.00%
  • Registros Duplicados     : 0

-------------------------------------------------------------------------------------
🛡️  CERTIFICACIÓN DE INTEGRIDAD REFERENCIAL:
  • Claves huerfanas residuales en Fact_Ventas: 0
  • Estado del Datamart                       : COMPLETO Y LISTO PARA CONEXIÓN A POWER BI
